<details>
   <summary><b>Table of Contents</b></summary>

   - [Note](#note)
   - [1. Review Data](#1-review-data)
   - [2. Data Inspection](#2-data-inspection)
     - [2.1. Missing Values Check](#21-missing-values-check)
     - [2.2. Duplicate Values Check](#22-duplicate-values-check)
     - [2.3. Data Types Check](#23-data-types-check)
     - [2.4. Unique Values Check](#24-unique-values-check)
     - [2.5. Numerical Values Check](#25-numerical-values-check)
     - [2.6. Categorical Values Check](#26-categorical-values-check)
   - [3. Data Cleaning](#3-data-cleaning)
     - [3.1. Job Title](#31-job-title)
     - [3.2. Salary Estimate](#32-salary-estimate)
     - [3.3. Job Description](#33-job-description)
     - [3.4. Rating](#34-rating)
     - [3.5. Company Name](#35-company-name)
     - [3.6. Location & Headquarters](#36-location--headquarters)
     - [3.7. Size](#37-size)
     - [3.8. Founded](#38-founded)
     - [3.9. Type of Ownership](#39-type-of-ownership)
     - [3.10. Industry](#310-industry)
     - [3.11. Sector](#311-sector)
     - [3.12. Revenue](#312-revenue)
     - [3.13. Competitors](#313-competitors)
   - [4. Feature Selection & Overview](#4-feature-selection--overview)

</details>

## **Note**
- **Data Source:** [Data Science Jobs & Salaries 2024](https://www.kaggle.com/datasets/fahadrehman07/data-science-jobs-and-salary-glassdoor?select=glassdoor_jobs.csv).
- **Objective:** This notebook focuses on data preprocessing, cleaning, and feature engineering to enhance the dataset for further analysis.

In [1]:
import re
import pandas as pd

# Config pandas
pd.set_option('display.max_columns', None) 

## **1. Review Data**
---

In [2]:
df = pd.read_csv('../data/glassdoor_jobs.csv') 
df = df.drop('Unnamed: 0', axis=1)
df.head()

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
0,Data Scientist,$53K-$91K (Glassdoor est.),"Data Scientist\r\nLocation: Albuquerque, NM\r\...",3.8,Tecolote Research\r\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),-1
1,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),What You Will Do:\r\n\r\nI. General Summary\r\...,3.4,University of Maryland Medical System\r\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),-1
2,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\r\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,Security Services,Business Services,$100 to $500 million (USD),-1
3,Data Scientist,$56K-$97K (Glassdoor est.),*Organization and Job ID**\r\nJob ID: 310709\r...,3.8,PNNL\r\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),"Oak Ridge National Laboratory, National Renewa..."
4,Data Scientist,$86K-$143K (Glassdoor est.),Data Scientist\r\nAffinity Solutions / Marketi...,2.9,Affinity Solutions\r\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee"


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 956 entries, 0 to 955
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Job Title          956 non-null    object 
 1   Salary Estimate    956 non-null    object 
 2   Job Description    956 non-null    object 
 3   Rating             956 non-null    float64
 4   Company Name       956 non-null    object 
 5   Location           956 non-null    object 
 6   Headquarters       956 non-null    object 
 7   Size               956 non-null    object 
 8   Founded            956 non-null    int64  
 9   Type of ownership  956 non-null    object 
 10  Industry           956 non-null    object 
 11  Sector             956 non-null    object 
 12  Revenue            956 non-null    object 
 13  Competitors        956 non-null    object 
dtypes: float64(1), int64(1), object(12)
memory usage: 104.7+ KB


## **2. Data Inspection**
---

### **2.1. Missing Values Check**

In [4]:
pd.DataFrame({
    'Column'        : df.columns,
    'Missing Values': df.isnull().sum().values
}).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
Column,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
Missing Values,0,0,0,0,0,0,0,0,0,0,0,0,0,0


**Observations**:

The dataset contains no missing values in any column, which is a positive indicator. However, verifying the validity and consistency of the data remains crucial. This will be addressed in the following sections.

### **2.2. Duplicate Values Check**

In [5]:
df[df.duplicated()==True]

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
30,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\r\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,Security Services,Business Services,$100 to $500 million (USD),-1
31,Data Scientist,$56K-$97K (Glassdoor est.),*Organization and Job ID**\r\nJob ID: 310709\r...,3.8,PNNL\r\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),"Oak Ridge National Laboratory, National Renewa..."
62,Data Scientist,$54K-$93K (Glassdoor est.),Job Description\r\n\r\n**Please only local can...,4.1,ClearOne Advantage\r\n4.1,"Baltimore, MD","Baltimore, MD",501 to 1000 employees,2008,Company - Private,Banks & Credit Unions,Finance,Unknown / Non-Applicable,-1
63,Data Scientist,$71K-$119K (Glassdoor est.),CyrusOne is seeking a talented Data Scientist ...,3.4,CyrusOne\r\n3.4,"Dallas, TX","Dallas, TX",201 to 500 employees,2000,Company - Public,Real Estate,Real Estate,$1 to $2 billion (USD),"Digital Realty, CoreSite, Equinix"
94,Staff Data Scientist - Technology,$106K-$172K (Glassdoor est.),Position Summary...\r\nDrives the execution of...,3.2,Walmart\r\n3.2,"Plano, TX","Bentonville, AR",10000+ employees,1962,Company - Public,"Department, Clothing, & Shoe Stores",Retail,$10+ billion (USD),"Target, Costco Wholesale, Amazon"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951,Senior Data Engineer,$72K-$133K (Glassdoor est.),THE CHALLENGE\r\nEventbrite has a world-class ...,4.4,Eventbrite\r\n4.4,"Nashville, TN","San Francisco, CA",1001 to 5000 employees,2006,Company - Public,Internet,Information Technology,$100 to $500 million (USD),"See Tickets, TicketWeb, Vendini"
952,"Project Scientist - Auton Lab, Robotics Institute",$56K-$91K (Glassdoor est.),The Auton Lab at Carnegie Mellon University is...,2.6,Software Engineering Institute\r\n2.6,"Pittsburgh, PA","Pittsburgh, PA",501 to 1000 employees,1984,College / University,Colleges & Universities,Education,Unknown / Non-Applicable,-1
953,Data Science Manager,$95K-$160K (Glassdoor est.),Data Science ManagerResponsibilities:\r\n\r\nO...,3.2,"Numeric, LLC\r\n3.2","Allentown, PA","Chadds Ford, PA",1 to 50 employees,-1,Company - Private,Staffing & Outsourcing,Business Services,$5 to $10 million (USD),-1
954,Data Engineer,-1,Loading...\r\n\r\nTitle: Data Engineer\r\n\r\n...,4.8,IGNW\r\n4.8,"Austin, TX","Portland, OR",201 to 500 employees,2015,Company - Private,IT Services,Information Technology,$25 to $50 million (USD),Slalom


In [6]:
# df[df.eq(df.iloc[62]).all(axis=1)]
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Number of duplicate rows: 356


**Observations**:

Analysis & Recommendations: The analysis identified 356 duplicated records, primarily consisting of job postings that were reposted multiple times (second or third postings). Handling duplicate data will follow two distinct approaches:

1. Retaining duplicates for recruitment trend analysis ([phase2 notebook](phase2_EDA.ipynb)):
   - Duplicate postings can provide valuable insights into hiring trends, such as which positions are frequently advertised or the periods when hiring demand increases.
   - Keeping these records allows for a more in-depth analysis of labor market trends and employer behavior.

2. Removing duplicates for salary prediction modeling ([phase3 notebook](phase3_modeling.ipynb)):
   - Repeated job postings do not contribute new information and may introduce bias, especially if the same company reposts the job with identical salary details.
   - Removing duplicates is crucial to ensure a diverse dataset, preventing model distortion and improving the accuracy of salary trend predictions.

### **2.3. Data Types Check**

In [7]:
pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values
}).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
Column,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
Data Type,object,object,object,float64,object,object,object,object,int64,object,object,object,object,object


### **2.4. Unique Values Check**

In [8]:
pd.DataFrame({
    'Column': df.columns,
    'Unique Values': [df[col].nunique() for col in df.columns]
}).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
Column,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
Unique Values,328,417,596,32,448,237,235,9,109,13,63,25,14,149


### **2.5. Numerical Values Check**

In [9]:
tmp = df[df.select_dtypes(include=['int64', 'float64']).columns].describe().T
tmp.index.name = 'Column'
tmp

,count,mean,std,min,25%,50%,75%,max
Column,,,,,,,,
Rating,956.0,3.601255,1.067619,-1.0,3.3,3.8,4.2,5.0
Founded,956.0,1774.605649,598.942517,-1.0,1937.0,1992.0,2008.0,2019.0


### **2.6. Categorical Values Check**

In [10]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
summary          = []

for col in categorical_cols:
    value_counts = df[col].value_counts().reset_index()
    value_counts.columns = ['Category', 'Count']
    value_counts['Column'] = col
    summary.append(value_counts[['Column', 'Category', 'Count']])

tmp = pd.concat(summary, ignore_index=True)
tmp

,Column,Category,Count
0,Job Title,Data Scientist,178
1,Job Title,Data Engineer,68
2,Job Title,Senior Data Scientist,42
3,Job Title,Data Analyst,18
4,Job Title,Senior Data Engineer,17
...,...,...,...
2529,Competitors,"Aquent, 24 Seven Talent",1
2530,Competitors,"MATRIX Resources, The Insource Group, TEKsystems",1
2531,Competitors,"CSC, IBM",1
2532,Competitors,"Ancestry, Verily Life Sciences, Abbott Laborat...",1


## **3. Data Cleaning**
---

### **3.1. `Job Title`**

Performing an in-depth analysis of unique values in the `Job Title` column to identify patterns and standardize job classifications. This process involves: 
- Grouping similar job roles into broader categories (`job_simplified`) to enhance data consistency and facilitate trend analysis. 
- Additionally, assessing the required experience level for each position allows for the classification of roles by seniority (`seniority`), enabling a more structured approach to workforce segmentation and salary prediction.

In [11]:
df['Job Title'].value_counts()

Job Title
Data Scientist                                                       178
Data Engineer                                                         68
Senior Data Scientist                                                 42
Data Analyst                                                          18
Senior Data Engineer                                                  17
                                                                    ... 
Jr. Data Scientist                                                     1
Data Architect / Data Modeler                                          1
Data Scientists                                                        1
Associate Scientist / Sr. Associate Scientist, Antibody Discovery      1
Machine Learning Engineer (NLP)                                        1
Name: count, Length: 328, dtype: int64

In [12]:
def simplify_title(title):
    if 'data scientist' in title.lower() or 'data science' in title.lower():
        return 'data scientist'
    elif 'data engineer' in title.lower():
        return 'data engineer'
    elif 'data analyst' in title.lower() or '(da)' in title.lower():
        return 'data analyst'
    elif 'machine learning' in title.lower() or 'ml' in title.lower():
        return 'machine learning engineer'
    elif 'software engineer' in title.lower() or 'developer' in title.lower():
        return 'software engineer'
    elif 'bi' in title.lower() or 'business intelligence' in title.lower():
        return 'business intelligence'
    elif 'research scientist' in title.lower():
        return 'research scientist'
    elif 'scientist' in title.lower():
        return 'other scientist'
    elif 'analytics' in title.lower():
        return 'other analytics'
    elif 'engineer' in title.lower():
        return 'other engineer'
    elif 'analytics' in title.lower():
        return 'relate analytics'
    elif 'specialist' in title.lower():
        return 'data specialist'
    else:
        return 'other'

df['job_simplified'] = df['Job Title'].apply(simplify_title)
df['job_simplified'].value_counts()

job_simplified
data scientist               401
data engineer                158
other scientist              140
data analyst                 109
business intelligence         46
machine learning engineer     28
other analytics               25
other                         19
research scientist            11
software engineer             10
data specialist                5
other engineer                 4
Name: count, dtype: int64

In [13]:
def extract_seniority(title):
    title = title.lower()
    if re.search(r'chief|vp|vice president|director|head', title):
      return 'Executive/Director'
    elif re.search(r'senior|sr|principal|lead|manager|managing|4|5', title):
        return 'Senior/Principal'
    elif re.search(r'associate|jr|junior|staff|ii|iii|2|3', title):
        return 'Associate'
    elif re.search(r'intern|entry|college hire|early career|graduate| i|1', title):
        return 'Entry Level'
    else:
        return 'Other'

df['seniority'] = df['Job Title'].apply(extract_seniority)
df['seniority'].value_counts()

seniority
Other                 501
Senior/Principal      307
Associate              71
Entry Level            39
Executive/Director     38
Name: count, dtype: int64

### **3.2. `Salary Estimate`**

The `Salary Estimate` variable contained **22.4% invalid values (-1)**, which were removed to maintain data integrity. To enhance salary analysis, job classification features (`Hourly`, `Employer_Provided`) were introduced, and salary figures were standardized. **Hourly wages** were converted to annual salaries using the **40-hour workweek, 52-week year standard**, ensuring consistency. Finally, `Min Salary`, `Max Salary`, and `Average Salary` were extracted to support predictive modeling.

In [14]:
df['Salary Estimate'].value_counts()

Salary Estimate
-1                                  214
$21-$34 Per Hour(Glassdoor est.)      6
$49K-$113K (Glassdoor est.)           6
$54K-$115K (Glassdoor est.)           6
$86K-$143K (Glassdoor est.)           6
                                   ... 
$105K-$173K (Glassdoor est.)          1
$46K-$85K (Glassdoor est.)            1
$71K-$134K (Glassdoor est.)           1
$102K-$190K (Glassdoor est.)          1
$27-$47 Per Hour(Glassdoor est.)      1
Name: count, Length: 417, dtype: int64

**Observations**:

As this is a key variable and contains **a significant proportion of missing or invalid values** (-1), accounting for 214 out of 956 records (~**22.4%**), traditional imputation methods may not yield accurate results. Therefore, the decision was made to remove rows containing this value to maintain data quality and analytical reliability.

In [15]:
df = df[df['Salary Estimate'] != '-1']

**Observations**:

To improve data clarity and facilitate analysis, several new variables related to job classification and salary structure are derived based on the information and distribution of values in the dataset.

Job Classification:

- `Hourly`: Identifies job postings where compensation is based on an hourly wage, typically associated with temporary or contract-based roles.
- `Employer_Provided`: Flags cases where salary details are explicitly provided by the employer, ensuring greater accuracy in compensation reporting.

Salary Standardization:

A widely recognized industry standard assumes **40 working hours per week and 52 weeks per year** when converting hourly wages to annual salaries. This assumption aligns with guidelines established by professional organizations such as the **Society for Human Resource Management (SHRM)** and the **U.S. Department of Labor**, ensuring consistency in salary comparisons ([Src 1](https://www.bls.gov/opub/hom/oews/calculation.htm), [Src 2](https://www.shrm.org/content/dam/en/shrm/topics-tools/tools/flsa-salary-increase-impact-analysis-guide-update.docx?downloadable=true)).

- Extracting Salary Ranges: Parsing the `Salary Estimate` column to derive `Min Salary` and `Max Salary`, representing the lower and upper bounds of compensation. 
- Adjusting Compensation for Hourly Roles: For jobs classified as hourly (`Hourly = 1`), converting their wages into an equivalent **annual salary** to align with full-time salary figures.
- Creating a Target Variable for Predictive Modeling: Computing `Average Salary` as the central measure of compensation, which will serve as the primary target variable for salary prediction models.

In [16]:
df['Hourly']            = df['Salary Estimate'].apply(lambda x: 1 if 'Per Hour' in x else 0)
df['employer_provided'] = df['Salary Estimate'].apply(lambda x: 1 if 'Employer Provided' in x else 0)

In [17]:
df['Salary Estimate Cleaned'] = df['Salary Estimate'].apply(lambda x: re.sub(r'[^\d\-]', '', x))

df['Min Salary'] = df.apply(
    lambda row: int(row['Salary Estimate Cleaned'].split('-')[0]) * 40 * 52 if row['Hourly'] == 1 else int(row['Salary Estimate Cleaned'].split('-')[0]) * 1000,
    axis=1
)

df['Max Salary'] = df.apply(
    lambda row: int(row['Salary Estimate Cleaned'].split('-')[1]) * 40 * 52 if row['Hourly'] == 1 else int(row['Salary Estimate Cleaned'].split('-')[1]) * 1000,
    axis=1
)

df['Average Salary'] = (df['Min Salary'] + df['Max Salary']) / 2

### **3.3. `Job Description`**

This section focuses on extracting **key skills** from the unstructured `Job Description` text. Given the inconsistencies in formatting and content, only relevant technical skills (e.g., Python, SQL, AWS, Excel, Tableau) are identified and encoded as binary variables for further analysis. Less relevant features are dropped to streamline the dataset.

In [18]:
df['Job Description']

0      Data Scientist\r\nLocation: Albuquerque, NM\r\...
1      What You Will Do:\r\n\r\nI. General Summary\r\...
2      KnowBe4, Inc. is a high growth information sec...
3      *Organization and Job ID**\r\nJob ID: 310709\r...
4      Data Scientist\r\nAffinity Solutions / Marketi...
                             ...                        
950    Site Name: USA - Massachusetts - Cambridge\r\n...
951    THE CHALLENGE\r\nEventbrite has a world-class ...
952    The Auton Lab at Carnegie Mellon University is...
953    Data Science ManagerResponsibilities:\r\n\r\nO...
955    Returning Candidate? Log back in to the Career...
Name: Job Description, Length: 742, dtype: object

**Observations**:

The `Job Description` field contains valuable details about job roles, including **title, location, education requirements, required skills, communication abilities, and benefits**.

However, it also presents several challenges:

- **Inconsistent data**: Variations in formatting and meaning make extraction and comparison difficult.
- **Unstructured text**: Job descriptions are free-form text, lacking a standardized format, which can result in missing or extraneous information during extraction.

To streamline the process, only the most **relevant** information—**required skills**—will be extracted for analysis.

In [19]:
df['Python_yn']            = df['Job Description'].apply(lambda x: 1 if 'python' in x.lower() else 0)
df['R Studio']             = df['Job Description'].apply(lambda x: 1 if 'r studio' in x.lower() or 'r-studio' in x.lower() or 'r_studio' in x.lower() else 0)
df['Spark']                = df['Job Description'].apply(lambda x: 1 if 'spark' in x.lower() else 0)
df['AWS_yn']               = df['Job Description'].apply(lambda x: 1 if 'aws' in x.lower() else 0)
df['Excel_yn']             = df['Job Description'].apply(lambda x: 1 if 'excel' in x.lower() else 0)
df['Tableau_yn']           = df['Job Description'].apply(lambda x: 1 if 'tableau' in x.lower() else 0)
df['PowerBI_yn']           = df['Job Description'].apply(lambda x: 1 if 'power bi' in x.lower() or 'powerbi' in x.lower() else 0)
df['MATLAB_yn']            = df['Job Description'].apply(lambda x: 1 if 'matlab' in x.lower() else 0)
df['SQL_yn']               = df['Job Description'].apply(lambda x: 1 if 'sql' in x.lower() else 0)
df['SAS_yn']               = df['Job Description'].apply(lambda x: 1 if 'sas' in x.lower() else 0)
df['MS_Access_yn']         = df['Job Description'].apply(lambda x: 1 if 'ms access' in x.lower() or 'microsoft access' in x.lower() else 0)
df['DataViz_yn']           = df['Job Description'].apply(lambda x: 1 if 'data visualization' in x.lower() or 'visualization' in x.lower() else 0)
df['Algorithmic_Aptitude'] = df['Job Description'].apply(lambda x: 1 if 'algorithmic aptitude' in x.lower() or 'algorithm' in x.lower() else 0)

In [20]:
# columns = [
#     ("Python", "Python_yn"), ("R Studio", "R Studio"), ("Spark", "Spark"), ("AWS", "AWS_yn"), ("Excel", "Excel_yn"),
#     ("Tableau", "Tableau_yn"), ("Power BI", "PowerBI_yn"), ("MATLAB", "MATLAB_yn"), ("SQL", "SQL_yn"), ("SAS", "SAS_yn"),
#     ("MS Access", "MS_Access_yn"), ("Data Visualization Tools", "DataViz_yn"), ("Algorithmic Aptitude", "Algorithmic_Aptitude")
# ]
# for name, col in columns:
#     print(f"{name}: {df[col].sum()}")
    
df = df.drop(columns=['R Studio', 'PowerBI_yn', 'MATLAB_yn', 'SAS_yn', 'MS_Access_yn', 'DataViz_yn', 'Algorithmic_Aptitude'])

### **3.4. `Rating`**

This section focuses on refining the `Rating` variable by categorizing its continuous values for better interpretability. **Missing ratings (-1) are assumed to represent newly established companies** with no reviews. A classification system is applied to group ratings into categories (`High`, `Medium`, `Low`, etc.), making the data more structured for analysis.

In [21]:
df['Rating'].unique()

array([ 3.8,  3.4,  4.8,  2.9,  4.1,  3.3,  4.6,  3.5,  3.2,  3.7,  3.6,
        3.9,  4.3,  4.2,  4. ,  4.7,  5. ,  3.1,  4.4,  2.8,  2.7,  1.9,
        4.5,  3. ,  2.3,  2.6, -1. ,  2.4,  2.5,  2.2,  2.1])

In [22]:
df[df['Rating']== -1]

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,job_simplified,seniority,Hourly,employer_provided,Salary Estimate Cleaned,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn
208,Principal Data Scientist with over 10 years ex...,Employer Provided Salary:$200K-$250K,Position Title: Principal Data Scientist\r\nLo...,-1.0,CA-One Tech Cloud,"San Francisco, CA","Fremont, CA",51 to 200 employees,2017,Company - Private,IT Services,Information Technology,$5 to $10 million (USD),-1,data scientist,Senior/Principal,0,1,200-250,200000,250000,225000.0,1,0,1,1,1,0
331,Principal Data Scientist with over 10 years ex...,Employer Provided Salary:$200K-$250K,Position Title: Principal Data Scientist\r\nLo...,-1.0,CA-One Tech Cloud,"San Francisco, CA","Fremont, CA",51 to 200 employees,2017,Company - Private,IT Services,Information Technology,$5 to $10 million (USD),-1,data scientist,Senior/Principal,0,1,200-250,200000,250000,225000.0,1,0,1,1,1,0
377,Data Operations Lead,Employer Provided Salary:$85K-$90K,Data Operations Lead\r\nLocation: Flexible tho...,-1.0,Muso,"San Francisco, CA","San Francisco, CA",201 to 500 employees,-1,Nonprofit Organization,-1,-1,Unknown / Non-Applicable,-1,other,Senior/Principal,0,1,85-90,85000,90000,87500.0,1,0,0,1,1,1
472,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",-1.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0
518,"Senior Scientist, Cell Pharmacology/Assay Deve...",Employer Provided Salary:$110K-$130K,"Senior Scientist, Cell Pharmacology/Assay Deve...",-1.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,other scientist,Senior/Principal,0,1,110-130,110000,130000,120000.0,0,0,0,0,0,0
583,Data Scientist,$81K-$140K (Glassdoor est.),"As a Data Scientist, you will play a critical ...",-1.0,ALIN,"New York, NY","Noida, India",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,data scientist,Other,0,0,81-140,81000,140000,110500.0,1,0,0,0,0,1
703,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",-1.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0
741,"Scientist – Cancer Discovery, Molecular Assay",Employer Provided Salary:$100K-$135K,"Scientist – Cancer Discovery, Molecular Assay\...",-1.0,Monte Rosa Therapeutics,"Cambridge, MA",-1,-1,-1,-1,-1,-1,-1,-1,other scientist,Other,0,1,100-135,100000,135000,117500.0,0,0,0,1,0,0
778,"Senior Scientist, Cell Pharmacology/Assay Deve...",Employer Provided Salary:$110K-$130K,"Senior Scientist, Cell Pharmacology/Assay Deve...",-1.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,other scientist,Senior/Principal,0,1,110-130,110000,130000,120000.0,0,0,0,0,0,0
819,"Principal Research Scientist/Team Lead, Medici...",Employer Provided Salary:$120K-$145K,"Principal Research Scientist/Team Lead, Medici...",-1.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,research scientist,Senior/Principal,0,1,120-145,120000,145000,132500.0,0,0,0,0,0,0


**Observations**:

- The `Rating` variable consists of continuous numerical values, making it suitable for classification into broader categories for better interpretability.

- Additionally, some records contain a **-1 value**, which often corresponds to companies where the `Founded` variable is also -1 (indicating an unknown founding year). Based on this pattern, it is reasonable to assume that **-1 represents newly established companies that have not yet received ratings**.

- Based on these insights, appropriate transformations will be applied to categorize and refine the `Rating` variable accordingly.

In [23]:
def classify_rating(rating):
    if rating == 0:
        return 'No Rating'
    if rating >=4:
        return 'High Rating'
    if rating >=3:
        return 'Medium Rating'
    if rating >=2:
        return 'Low Rating'
    else:
        return 'Very Low Rating'

df['Rating']          = df['Rating'].replace(-1, 0)
df['Rating Category'] = df['Rating'].apply(classify_rating)
df['Rating Category'].value_counts()

Rating Category
Medium Rating      440
High Rating        219
Low Rating          69
No Rating           11
Very Low Rating      3
Name: count, dtype: int64

### **3.5. `Company Name`**

Simply removes unnecessary string, ensuring cleaner and more standardized entries.

In [24]:
df['Company Name']

0                          Tecolote Research\r\n3.8
1      University of Maryland Medical System\r\n3.4
2                                    KnowBe4\r\n4.8
3                                       PNNL\r\n3.8
4                         Affinity Solutions\r\n2.9
                           ...                     
950                                      GSK\r\n3.9
951                               Eventbrite\r\n4.4
952           Software Engineering Institute\r\n2.6
953                             Numeric, LLC\r\n3.2
955             Riverside Research Institute\r\n3.6
Name: Company Name, Length: 742, dtype: object

**Observations**:

- Each company name is followed by a redundant combination of `"\r"` and the corresponding `Rating` value, which does not add meaningful information.
- To clean the data, this **"\r" and any text following it** will be removed from all entries in the `Company Name` variable.

In [25]:
df['Company Name'] = df['Company Name'].str.split('\r').str[0]
df['Company Name']

0                          Tecolote Research
1      University of Maryland Medical System
2                                    KnowBe4
3                                       PNNL
4                         Affinity Solutions
                       ...                  
950                                      GSK
951                               Eventbrite
952           Software Engineering Institute
953                             Numeric, LLC
955             Riverside Research Institute
Name: Company Name, Length: 742, dtype: object

### **3.6. `Location` & `Headquarters`**

Extracts **job state** from `Location` and creates a binary variable to indicate whether the **job location and company headquarters are in the same state**, enhancing geographic insights.

In [26]:
df['Location']

0      Albuquerque, NM
1        Linthicum, MD
2       Clearwater, FL
3         Richland, WA
4         New York, NY
            ...       
950      Cambridge, MA
951      Nashville, TN
952     Pittsburgh, PA
953      Allentown, PA
955    Beavercreek, OH
Name: Location, Length: 742, dtype: object

**Observations**:

To enhance geographic information, new variables are derived from existing location-related columns (`Location` and `Headquarters`):

- `job_state`: Extracts the state from the `Location` variable to standardize job location data.
- `same_state`: A binary variable indicating whether the job location (`Location`) and company headquarters (`Headquarters`) are in the same state, providing insights into company expansion patterns.

In [27]:
df['job_state']  = df['Location'].apply(lambda x: x.split(',')[1])
df['same_state'] = df.apply(lambda x: 1 if x.Location==x.Headquarters else 0, axis =1)

### **3.7. `Size`**

Standardizes company size data for consistency.

In [28]:
df['Size'].unique()

array(['501 to 1000 employees', '10000+ employees',
       '1001 to 5000 employees', '51 to 200 employees',
       '201 to 500 employees', '5001 to 10000 employees',
       '1 to 50 employees', 'Unknown', '-1'], dtype=object)

In [29]:
df[df['Size'].isin(['-1', 'Unknown'])]

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,job_simplified,seniority,Hourly,employer_provided,Salary Estimate Cleaned,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn,Rating Category,job_state,same_state
48,Data Scientist,Employer Provided Salary:$150K-$160K,"BPA Services, LLC is seeking a Computer/Data S...",5.0,BPA Services,"Washington, DC","Alexandria, VA",Unknown,-1,Company - Private,Enterprise Software & Network Solutions,Information Technology,Unknown / Non-Applicable,-1,data scientist,Other,0,1,150-160,150000,160000,155000.0,0,0,1,1,0,0,High Rating,DC,0
472,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0
477,Data Scientist,Employer Provided Salary:$150K-$160K,"BPA Services, LLC is seeking a Computer/Data S...",5.0,BPA Services,"Washington, DC","Alexandria, VA",Unknown,-1,Company - Private,Enterprise Software & Network Solutions,Information Technology,Unknown / Non-Applicable,-1,data scientist,Other,0,1,150-160,150000,160000,155000.0,0,0,1,1,0,0,High Rating,DC,0
518,"Senior Scientist, Cell Pharmacology/Assay Deve...",Employer Provided Salary:$110K-$130K,"Senior Scientist, Cell Pharmacology/Assay Deve...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,other scientist,Senior/Principal,0,1,110-130,110000,130000,120000.0,0,0,0,0,0,0,No Rating,MA,0
583,Data Scientist,$81K-$140K (Glassdoor est.),"As a Data Scientist, you will play a critical ...",0.0,ALIN,"New York, NY","Noida, India",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,data scientist,Other,0,0,81-140,81000,140000,110500.0,1,0,0,0,0,1,No Rating,NY,0
703,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0
741,"Scientist – Cancer Discovery, Molecular Assay",Employer Provided Salary:$100K-$135K,"Scientist – Cancer Discovery, Molecular Assay\...",0.0,Monte Rosa Therapeutics,"Cambridge, MA",-1,-1,-1,-1,-1,-1,-1,-1,other scientist,Other,0,1,100-135,100000,135000,117500.0,0,0,0,1,0,0,No Rating,MA,0
778,"Senior Scientist, Cell Pharmacology/Assay Deve...",Employer Provided Salary:$110K-$130K,"Senior Scientist, Cell Pharmacology/Assay Deve...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,other scientist,Senior/Principal,0,1,110-130,110000,130000,120000.0,0,0,0,0,0,0,No Rating,MA,0
819,"Principal Research Scientist/Team Lead, Medici...",Employer Provided Salary:$120K-$145K,"Principal Research Scientist/Team Lead, Medici...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,research scientist,Senior/Principal,0,1,120-145,120000,145000,132500.0,0,0,0,0,0,0,No Rating,MA,0
943,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0


**Observations**:

To maintain consistency, the value **"-1"** is replaced with **"Unknown"**, as it appears only once in the dataset.

In [30]:
df['Size']=df['Size'].replace('-1','Unknown')

### **3.8. `Founded`**

Retains **"-1"** values in `Founded` for potential imputation and introduces the `Age` variable, which converts the founding year into company age for more meaningful analysis.

In [31]:
df['Founded'].unique()

array([1973, 1984, 2010, 1965, 1998, 2000, 2008, 2005, 2014, 2009, 2011,
       1968, 1962, 2012, 1781, 1995, 1915, 2013, 1935, 1849, 1952, 1852,
       1997, 1996, 1974, 1969, 1870, 1985,   -1, 2015, 1993, 1958, 1986,
       1999, 1925, 1912, 2002, 1863, 1939, 2016, 1885, 2006, 1948, 2003,
       1927, 1978, 1860, 2017, 1942, 1990, 1988, 2001, 2007, 1992, 1994,
       1977, 2019, 1982, 1937, 1878, 1966, 1971, 1943, 1987, 1945, 1846,
       1851, 1976, 1981, 1970, 1951, 1967, 1961, 1964, 1930, 1917, 1883,
       1887, 2004, 1850, 1902, 1744, 1929, 1947, 1991, 1989, 1928, 1875,
       1913, 1972, 1856, 1983, 1922, 1812, 1914, 1980, 1954, 1830, 1975,
       1899, 1979, 1889])

In [32]:
df[df['Founded']==-1]

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,job_simplified,seniority,Hourly,employer_provided,Salary Estimate Cleaned,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn,Rating Category,job_state,same_state
43,Data Engineer,$68K-$129K (Glassdoor est.),Position Title: Data Engineer\r\n\r\nPersivia ...,3.6,Persivia,"Marlborough, MA","Lowell, MA",1 to 50 employees,-1,Company - Private,-1,-1,Less than $1 million (USD),-1,data engineer,Other,0,0,68-129,68000,129000,98500.0,0,0,0,1,0,1,Medium Rating,MA,0
48,Data Scientist,Employer Provided Salary:$150K-$160K,"BPA Services, LLC is seeking a Computer/Data S...",5.0,BPA Services,"Washington, DC","Alexandria, VA",Unknown,-1,Company - Private,Enterprise Software & Network Solutions,Information Technology,Unknown / Non-Applicable,-1,data scientist,Other,0,1,150-160,150000,160000,155000.0,0,0,1,1,0,0,High Rating,DC,0
76,Data Scientist,$96K-$161K (Glassdoor est.),SummaryProvide data management and statistical...,3.2,"Numeric, LLC","Philadelphia, PA","Chadds Ford, PA",1 to 50 employees,-1,Company - Private,Staffing & Outsourcing,Business Services,$5 to $10 million (USD),-1,data scientist,Other,0,0,96-161,96000,161000,128500.0,1,1,1,0,0,1,Medium Rating,PA,0
166,Data Engineer 4 - Contract,$59K-$115K (Glassdoor est.),Purposes\r\n\r\nAs a member of the Business In...,4.2,The Church of Jesus Christ of Latter-day Saints,"Riverton, UT","Salt Lake City, UT",10000+ employees,-1,Nonprofit Organization,Religious Organizations,Non-Profit,Unknown / Non-Applicable,-1,data engineer,Senior/Principal,0,0,59-115,59000,115000,87000.0,0,0,1,1,0,0,High Rating,UT,0
192,Data Engineer 5 - Contract (Remote),$74K-$140K (Glassdoor est.),Purposes\r\n\r\nThis is a remote contract posi...,4.2,The Church of Jesus Christ of Latter-day Saints,"Riverton, UT","Salt Lake City, UT",10000+ employees,-1,Nonprofit Organization,Religious Organizations,Non-Profit,Unknown / Non-Applicable,-1,data engineer,Senior/Principal,0,0,74-140,74000,140000,107000.0,0,0,0,1,0,0,High Rating,UT,0
223,PV Scientist,$60K-$123K (Glassdoor est.),SUMMARY:\r\nThe Pharmacovigilance Scientist su...,2.9,Karyopharm Therapeutics Inc.,"Newton, MA","Newton, MA",201 to 500 employees,-1,Company - Public,Biotech & Pharmaceuticals,Biotech & Pharmaceuticals,Unknown / Non-Applicable,-1,other scientist,Other,0,0,60-123,60000,123000,91500.0,0,0,0,1,0,0,Low Rating,MA,1
226,Data Engineer,$48K-$93K (Glassdoor est.),Are you interested in a career opportunity wit...,3.7,P2 Energy Solutions,"Lafayette, LA","Denver, CO",501 to 1000 employees,-1,Company - Private,Computer Hardware & Software,Information Technology,Unknown / Non-Applicable,-1,data engineer,Other,0,0,48-93,48000,93000,70500.0,0,0,0,1,0,1,Medium Rating,LA,0
242,R&D Specialist/ Food Scientist,$39K-$66K (Glassdoor est.),Responsibilities Include but may not be limite...,2.4,Teasdale Latin Foods,"Hoopeston, IL","Flower Mound, TX",501 to 1000 employees,-1,Company - Private,Food & Beverage Manufacturing,Manufacturing,$100 to $500 million (USD),-1,other scientist,Other,0,0,39-66,39000,66000,52500.0,0,0,0,0,0,0,Low Rating,IL,0
270,Data Analyst,$33K-$62K (Glassdoor est.),"Wednesday, March 11, 2020\r\n\r\n\r\nCommunity...",2.8,Community Action Partnership of San Luis Obispo,"Parlier, CA","San Luis Obispo, CA",501 to 1000 employees,-1,Nonprofit Organization,Social Assistance,Non-Profit,$50 to $100 million (USD),-1,data analyst,Other,0,0,33-62,33000,62000,47500.0,0,0,0,1,0,0,Low Rating,CA,0
282,Data Engineer 5 - Contract (Remote),$74K-$140K (Glassdoor est.),Purposes\r\n\r\nThis is a remote contract posi...,4.2,The Church of Jesus Christ of Latter-day Saints,"Riverton, UT","Salt Lake City, UT",10000+ employees,-1,Nonprofit Organization,Religious Organizations,Non-Profit,Unknown / Non-Applicable,-1,data engineer,Senior/Principal,0,0,74-140,74000,140000,107000.0,0,0,0,1,0,0,High Rating,U

**Observations**:

- The **"-1"** value in the `Founded` variable likely results from data entry errors or missing information. Since it appears in approximately **50 rows (~5%)**, removal is not the preferred approach. Instead, it will be retained for potential imputation after visualizing its distribution or assumed to represent newly established companies.

- A new variable, **Age**, is created by transforming **Founded** into the company's age, as this provides more meaningful insights for analysis compared to the founding year.

In [33]:
df['Age'] = df['Founded'].apply(lambda x: x if x<1 else 2023-x)
df['Age'].value_counts()

Age
-1      50
 13     32
 15     31
 27     27
 17     24
        ..
 211     1
 109     1
 124     1
 44      1
 134     1
Name: count, Length: 102, dtype: int64

### **3.9. `Type of ownership`**

Standardizes data for consistency.

In [34]:
df['Type of ownership'].unique()

array(['Company - Private', 'Other Organization', 'Government',
       'Company - Public', 'Hospital', 'Subsidiary or Business Segment',
       'Nonprofit Organization', 'Unknown', 'College / University',
       'School / School District', '-1'], dtype=object)

In [35]:
df[df['Type of ownership'].isin(['-1', 'Unknown'])]

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,job_simplified,seniority,Hourly,employer_provided,Salary Estimate Cleaned,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn,Rating Category,job_state,same_state,Age
143,Project Scientist,$29K-$50K (Glassdoor est.),Project Scientist\r\n\r\nJob Details\r\nLevel\...,4.0,Alliance Source Testing,"Alabaster, AL","Decatur, AL",51 to 200 employees,2000,Unknown,Architectural & Engineering Services,Business Services,$25 to $50 million (USD),-1,other scientist,Other,0,0,29-50,29000,50000,39500.0,0,0,0,0,0,0,High Rating,AL,0,23
741,"Scientist – Cancer Discovery, Molecular Assay",Employer Provided Salary:$100K-$135K,"Scientist – Cancer Discovery, Molecular Assay\...",0.0,Monte Rosa Therapeutics,"Cambridge, MA",-1,Unknown,-1,-1,-1,-1,-1,-1,other scientist,Other,0,1,100-135,100000,135000,117500.0,0,0,0,1,0,0,No Rating,MA,0,-1


**Observations**:

To maintain consistency, the value **"-1"** is replaced with **"Unknown"**.

In [36]:
df['Type of ownership'] = df['Type of ownership'].replace('-1','Unknown')

### **3.10. `Industry`**

Standardizes data for consistency.

In [37]:
df['Industry'].unique()

array(['Aerospace & Defense', 'Health Care Services & Hospitals',
       'Security Services', 'Energy', 'Advertising & Marketing',
       'Real Estate', 'Banks & Credit Unions', 'Consulting', 'Internet',
       'Other Retail Stores', 'Research & Development',
       'Department, Clothing, & Shoe Stores', 'Biotech & Pharmaceuticals',
       'Motion Picture Production & Distribution',
       'Enterprise Software & Network Solutions', 'Insurance Carriers',
       'Insurance Agencies & Brokerages', 'Logistics & Supply Chain',
       'Telecommunications Services', 'IT Services',
       'Computer Hardware & Software', '-1',
       'Consumer Products Manufacturing', 'Industrial Manufacturing',
       'Metals Brokers', 'Financial Transaction Processing',
       'Sporting Goods Stores', 'Staffing & Outsourcing', 'Wholesale',
       'Mining', 'Financial Analytics & Research', 'Federal Agencies',
       'Education Training Services',
       'Transportation Equipment Manufacturing', 'Farm Support 

In [38]:
df[df['Industry'].isin(['-1', 'Unknown'])]

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,job_simplified,seniority,Hourly,employer_provided,Salary Estimate Cleaned,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn,Rating Category,job_state,same_state,Age
43,Data Engineer,$68K-$129K (Glassdoor est.),Position Title: Data Engineer\r\n\r\nPersivia ...,3.6,Persivia,"Marlborough, MA","Lowell, MA",1 to 50 employees,-1,Company - Private,-1,-1,Less than $1 million (USD),-1,data engineer,Other,0,0,68-129,68000,129000,98500.0,0,0,0,1,0,1,Medium Rating,MA,0,-1
377,Data Operations Lead,Employer Provided Salary:$85K-$90K,Data Operations Lead\r\nLocation: Flexible tho...,0.0,Muso,"San Francisco, CA","San Francisco, CA",201 to 500 employees,-1,Nonprofit Organization,-1,-1,Unknown / Non-Applicable,-1,other,Senior/Principal,0,1,85-90,85000,90000,87500.0,1,0,0,1,1,1,No Rating,CA,1,-1
472,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0,-1
518,"Senior Scientist, Cell Pharmacology/Assay Deve...",Employer Provided Salary:$110K-$130K,"Senior Scientist, Cell Pharmacology/Assay Deve...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,other scientist,Senior/Principal,0,1,110-130,110000,130000,120000.0,0,0,0,0,0,0,No Rating,MA,0,-1
583,Data Scientist,$81K-$140K (Glassdoor est.),"As a Data Scientist, you will play a critical ...",0.0,ALIN,"New York, NY","Noida, India",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,data scientist,Other,0,0,81-140,81000,140000,110500.0,1,0,0,0,0,1,No Rating,NY,0,-1
703,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0,-1
741,"Scientist – Cancer Discovery, Molecular Assay",Employer Provided Salary:$100K-$135K,"Scientist – Cancer Discovery, Molecular Assay\...",0.0,Monte Rosa Therapeutics,"Cambridge, MA",-1,Unknown,-1,Unknown,-1,-1,-1,-1,other scientist,Other,0,1,100-135,100000,135000,117500.0,0,0,0,1,0,0,No Rating,MA,0,-1
778,"Senior Scientist, Cell Pharmacology/Assay Deve...",Employer Provided Salary:$110K-$130K,"Senior Scientist, Cell Pharmacology/Assay Deve...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,other scientist,Senior/Principal,0,1,110-130,110000,130000,120000.0,0,0,0,0,0,0,No Rating,MA,0,-1
819,"Principal Research Scientist/Team Lead, Medici...",Employer Provided Salary:$120K-$145K,"Principal Research Scientist/Team Lead, Medici...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,research scientist,Senior/Principal,0,1,120-145,120000,145000,132500.0,0,0,0,0,0,0,No Rating,MA,0,-1
943,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0,-1


**Observations**:

To maintain consistency, the value **"-1"** is replaced with **"Unknown"**.

In [39]:
df['Industry'] = df['Industry'].replace('-1', 'Unknown')

### **3.11. `Sector`**

Standardizes its data for consistency, and clarifies its distinction from `Industry`.

In [40]:
df['Sector'].unique()

array(['Aerospace & Defense', 'Health Care', 'Business Services',
       'Oil, Gas, Energy & Utilities', 'Real Estate', 'Finance',
       'Information Technology', 'Retail', 'Biotech & Pharmaceuticals',
       'Media', 'Insurance', 'Transportation & Logistics',
       'Telecommunications', '-1', 'Manufacturing', 'Mining & Metals',
       'Government', 'Education', 'Agriculture & Forestry',
       'Travel & Tourism', 'Non-Profit',
       'Arts, Entertainment & Recreation',
       'Construction, Repair & Maintenance', 'Accounting & Legal',
       'Consumer Services'], dtype=object)

In [41]:
df[df['Sector'].isin(['-1', 'Unknown'])]

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,job_simplified,seniority,Hourly,employer_provided,Salary Estimate Cleaned,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn,Rating Category,job_state,same_state,Age
43,Data Engineer,$68K-$129K (Glassdoor est.),Position Title: Data Engineer\r\n\r\nPersivia ...,3.6,Persivia,"Marlborough, MA","Lowell, MA",1 to 50 employees,-1,Company - Private,Unknown,-1,Less than $1 million (USD),-1,data engineer,Other,0,0,68-129,68000,129000,98500.0,0,0,0,1,0,1,Medium Rating,MA,0,-1
377,Data Operations Lead,Employer Provided Salary:$85K-$90K,Data Operations Lead\r\nLocation: Flexible tho...,0.0,Muso,"San Francisco, CA","San Francisco, CA",201 to 500 employees,-1,Nonprofit Organization,Unknown,-1,Unknown / Non-Applicable,-1,other,Senior/Principal,0,1,85-90,85000,90000,87500.0,1,0,0,1,1,1,No Rating,CA,1,-1
472,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,Unknown,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0,-1
518,"Senior Scientist, Cell Pharmacology/Assay Deve...",Employer Provided Salary:$110K-$130K,"Senior Scientist, Cell Pharmacology/Assay Deve...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,Unknown,-1,Unknown / Non-Applicable,-1,other scientist,Senior/Principal,0,1,110-130,110000,130000,120000.0,0,0,0,0,0,0,No Rating,MA,0,-1
583,Data Scientist,$81K-$140K (Glassdoor est.),"As a Data Scientist, you will play a critical ...",0.0,ALIN,"New York, NY","Noida, India",Unknown,-1,Company - Private,Unknown,-1,Unknown / Non-Applicable,-1,data scientist,Other,0,0,81-140,81000,140000,110500.0,1,0,0,0,0,1,No Rating,NY,0,-1
703,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,Unknown,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0,-1
741,"Scientist – Cancer Discovery, Molecular Assay",Employer Provided Salary:$100K-$135K,"Scientist – Cancer Discovery, Molecular Assay\...",0.0,Monte Rosa Therapeutics,"Cambridge, MA",-1,Unknown,-1,Unknown,Unknown,-1,-1,-1,other scientist,Other,0,1,100-135,100000,135000,117500.0,0,0,0,1,0,0,No Rating,MA,0,-1
778,"Senior Scientist, Cell Pharmacology/Assay Deve...",Employer Provided Salary:$110K-$130K,"Senior Scientist, Cell Pharmacology/Assay Deve...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,Unknown,-1,Unknown / Non-Applicable,-1,other scientist,Senior/Principal,0,1,110-130,110000,130000,120000.0,0,0,0,0,0,0,No Rating,MA,0,-1
819,"Principal Research Scientist/Team Lead, Medici...",Employer Provided Salary:$120K-$145K,"Principal Research Scientist/Team Lead, Medici...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,Unknown,-1,Unknown / Non-Applicable,-1,research scientist,Senior/Principal,0,1,120-145,120000,145000,132500.0,0,0,0,0,0,0,No Rating,MA,0,-1
943,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,Unknown,-1,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0,-1


**Observations**:

- To maintain consistency, the value **"-1"** is replaced with **"Unknown"**.

- Comparing `Sector` and `Industry`:
    - `Sector` represents broader categories, while `Industry` provides more specific classifications. This distinction helps in understanding different market segments.
    - Since the analysis focuses on a **high-level overview**, no further classification or grouping is applied.

In [42]:
df['Sector'] = df['Sector'].replace('-1', 'Unknown')

### **3.12. `Revenue`**

Standardizes data for consistency.

In [43]:
df['Revenue'].unique()

array(['$50 to $100 million (USD)', '$2 to $5 billion (USD)',
       '$100 to $500 million (USD)', '$500 million to $1 billion (USD)',
       'Unknown / Non-Applicable', '$1 to $2 billion (USD)',
       '$25 to $50 million (USD)', '$10+ billion (USD)',
       '$1 to $5 million (USD)', '$10 to $25 million (USD)',
       '$5 to $10 billion (USD)', 'Less than $1 million (USD)',
       '$5 to $10 million (USD)', '-1'], dtype=object)

In [44]:
df[df['Revenue'].isin(['-1', 'Unknown / Non-Applicable'])]

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,job_simplified,seniority,Hourly,employer_provided,Salary Estimate Cleaned,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn,Rating Category,job_state,same_state,Age
4,Data Scientist,$86K-$143K (Glassdoor est.),Data Scientist\r\nAffinity Solutions / Marketi...,2.9,Affinity Solutions,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee",data scientist,Other,0,0,86-143,86000,143000,114500.0,1,0,0,1,0,1,Low Rating,NY,1,25
6,Data Scientist,$54K-$93K (Glassdoor est.),Job Description\r\n\r\n**Please only local can...,4.1,ClearOne Advantage,"Baltimore, MD","Baltimore, MD",501 to 1000 employees,2008,Company - Private,Banks & Credit Unions,Finance,Unknown / Non-Applicable,-1,data scientist,Other,0,0,54-93,54000,93000,73500.0,0,0,0,1,0,0,High Rating,MD,1,15
13,Data Analyst,$46K-$85K (Glassdoor est.),"Are you an experienced Data Analyst, skilled a...",4.1,Yesler,"Seattle, WA","Seattle, WA",201 to 500 employees,2012,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,-1,data analyst,Other,0,0,46-85,46000,85000,65500.0,1,1,1,1,1,1,High Rating,WA,1,11
15,Data Engineer I,$102K-$190K (Glassdoor est.),This opportunity is within Audibles Data Engin...,3.6,Audible,"Newark, NJ","Newark, NJ",1001 to 5000 employees,1995,Subsidiary or Business Segment,Motion Picture Production & Distribution,Media,Unknown / Non-Applicable,-1,data engineer,Entry Level,0,0,102-190,102000,190000,146000.0,0,0,0,1,0,0,Medium Rating,NJ,1,28
17,Customer Data Scientist,$118K-$189K (Glassdoor est.),Company Overview\r\n\r\nH2O.ai is the open sou...,4.3,h2o.ai,"Mountain View, CA","Mountain View, CA",201 to 500 employees,2011,Company - Private,Enterprise Software & Network Solutions,Information Technology,Unknown / Non-Applicable,-1,data scientist,Other,0,0,118-189,118000,189000,153500.0,1,1,1,1,0,0,High Rating,CA,1,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
942,"Associate Scientist, LC/MS Biologics",$44K-$96K (Glassdoor est.),Q2 Solutions is a leading clinical trial labor...,2.9,Q2 Solutions,"Ithaca, NY","Morrisville, NC",1001 to 5000 employees,2015,Company - Private,Biotech & Pharmaceuticals,Biotech & Pharmaceuticals,Unknown / Non-Applicable,-1,business intelligence,Associate,0,0,44-96,44000,96000,70000.0,0,0,0,1,0,0,Low Rating,NY,0,8
943,"Research Scientist, Immunology - Cancer Biology",Employer Provided Salary:$100K-$140K,"Research Scientist, Immunology - Cancer Biolog...",0.0,Kronos Bio,"Cambridge, MA","San Mateo, CA",Unknown,-1,Company - Private,Unknown,Unknown,Unknown / Non-Applicable,-1,business intelligence,Entry Level,0,1,100-140,100000,140000,120000.0,0,0,0,1,0,0,No Rating,MA,0,-1
945,Machine Learning Engineer (NLP),$80K-$142K (Glassdoor est.),CK-12’s mission is to provide free access to o...,4.1,CK-12 Foundation,"Palo Alto, CA","Palo Alto, CA",1 to 50 employees,2007,Company - Private,K-12 Education,Education,Unknown / Non-Applicable,-1,machine learning engineer,Other,0,0,80-142,80000,142000,111000.0,1,0,1,1,0,0,High Rating,CA,1,16
946,Senior Data Analyst,$99K-$178K (Glassdoor est.),Senior Data Analyst\r\n\r\nAbout us\r\n\r\n\r\...,3.9,Life360,"San Francisco, CA","San Francisco, CA",51 to 200 employees,2008,Company - Public,Computer Hardware & Software,Information Technology,Unknown / Non-Applicable,-1,data analyst,Senior/Principal,0,0,99-178,99000,178000,138500.0,1,0,0,0,1,1,Medium Rating,CA,1,15


**Observations**:

To maintain consistency, the value **"-1"** and **"Unknown / Non-Applicable"** is replaced with **"Unknown"**.

In [45]:
df['Revenue'] = df['Revenue'].replace(['-1', 'Unknown / Non-Applicable'], 'Unknown')

### **3.13. `Competitors`**

Simply remove this variable, as it not relevant to the objective of this project.

In [46]:
df['Competitors'].unique()

array(['-1',
       'Oak Ridge National Laboratory, National Renewable Energy Lab, Los Alamos National Laboratory',
       'Commerce Signals, Cardlytics, Yodlee',
       'Digital Realty, CoreSite, Equinix', 'Clicktripz, SmarterTravel',
       'Target, Costco Wholesale, Amazon', 'Novartis, Baxter, Pfizer',
       'bluebird bio, Agios Pharmaceuticals, Celgene',
       "Angie's List, HomeAdvisor, Thumbtack",
       'Leidos, CACI International, Booz Allen Hamilton',
       'Thermo Fisher Scientific, Enzymatics, Illumina', 'Pitney Bowes',
       'BrowserStack, Selenium Master, Perfecto Mobile',
       'Unilever, Procter & Gamble, Henkel',
       'UDR, AvalonBay Communities, Essex Property Trust',
       'American Express, Mastercard, Discover',
       'TASC, Vencore, Booz Allen Hamilton',
       'John Deere, Komatsu, CNH Industrial',
       'Travelers, Allstate, State Farm', 'Munich Re, Hannover RE, SCOR',
       'Skyhigh Networks, Zscaler, NortonLifeLock',
       'Slalom, Daugherty Busines

**Observations**:

Since the primary focus of this analysis is **salary trends in the data industry**, the number of competitors is not a highly relevant factor. Therefore, the `Competitors` variable is removed from the dataset.

In [47]:
df = df.drop(columns=['Competitors'])

## **4. Feature Selection & Overview**
---

In [48]:
df.head()

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,job_simplified,seniority,Hourly,employer_provided,Salary Estimate Cleaned,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn,Rating Category,job_state,same_state,Age
0,Data Scientist,$53K-$91K (Glassdoor est.),"Data Scientist\r\nLocation: Albuquerque, NM\r\...",3.8,Tecolote Research,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),data scientist,Other,0,0,53-91,53000,91000,72000.0,1,0,0,1,1,0,Medium Rating,NM,0,50
1,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),What You Will Do:\r\n\r\nI. General Summary\r\...,3.4,University of Maryland Medical System,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),data scientist,Other,0,0,63-112,63000,112000,87500.0,1,0,0,0,0,0,Medium Rating,MD,0,39
2,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,Security Services,Business Services,$100 to $500 million (USD),data scientist,Other,0,0,80-90,80000,90000,85000.0,1,1,0,1,0,1,High Rating,FL,1,13
3,Data Scientist,$56K-$97K (Glassdoor est.),*Organization and Job ID**\r\nJob ID: 310709\r...,3.8,PNNL,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),data scientist,Other,0,0,56-97,56000,97000,76500.0,1,0,0,0,0,0,Medium Rating,WA,1,58
4,Data Scientist,$86K-$143K (Glassdoor est.),Data Scientist\r\nAffinity Solutions / Marketi...,2.9,Affinity Solutions,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown,data scientist,Other,0,0,86-143,86000,143000,114500.0,1,0,0,1,0,1,Low Rating,NY,1,25


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 742 entries, 0 to 955
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Job Title                742 non-null    object 
 1   Salary Estimate          742 non-null    object 
 2   Job Description          742 non-null    object 
 3   Rating                   742 non-null    float64
 4   Company Name             742 non-null    object 
 5   Location                 742 non-null    object 
 6   Headquarters             742 non-null    object 
 7   Size                     742 non-null    object 
 8   Founded                  742 non-null    int64  
 9   Type of ownership        742 non-null    object 
 10  Industry                 742 non-null    object 
 11  Sector                   742 non-null    object 
 12  Revenue                  742 non-null    object 
 13  job_simplified           742 non-null    object 
 14  seniority                742 no

To refine the dataset and retain only the most relevant features for analysis, unnecessary columns are removed based on redundancy, limited utility, and alignment with the research objectives.

Columns Removed and Justifications:

| **Column**                 | **Rationale**                                                                                  | **Action**     |
|----------------------------|----------------------------------------------------------------------------------------------|---------------|
| **`Job Title`**            | Too many unique values; simplified into `job_simplified` for better categorization.         | Removed       |
| **`Salary Estimate`**      | Contains unstandardized salary data; replaced with `Min Salary`, `Max Salary`, and `Average Salary`. | Removed       |
| **`Job Description`**      | Unstructured text; extracting meaningful insights requires NLP processing, which is out of scope. | Removed       |
| **`Founded`**              | The `Age` variable provides a more meaningful representation of company tenure.              | Removed       |
| **`employer_provided`**    | Limited analytical impact unless a specific study on employer-provided salaries is conducted. | Removed       |
| **`Hourly`**               | Retained only if analyzing hourly wage trends; otherwise, removed.                          | Conditional   |
| **`Salary Estimate Cleaned`** | Redundant as salary-related variables (`Min Salary`, `Max Salary`, `Average Salary`) are already included. | Removed       |

Key Reasons for Removal:

1. **Redundancy**: Columns like `Salary Estimate Cleaned`, `Rating Category`, and `job_state` have more refined or representative counterparts.
2. **Limited Direct Utility**: `Job Description` requires complex text analysis for insights, making it less relevant without NLP techniques.
3. **Narrow Scope**: `Hourly` and `employer_provided` may not contribute meaningfully unless a specific focus on wage structure is required.
4. **Streamlining the Dataset**: Removing unnecessary columns enhances processing efficiency and focuses the analysis on key salary-related factors.

In [50]:
columns_to_drop_visualization = ['Job Title', 'Salary Estimate', 'Job Description', 'Founded', 'employer_provided', 'Hourly', 'Salary Estimate Cleaned']

data_visualization = df.drop(columns=columns_to_drop_visualization)
data_visualization.to_csv('../data/data_EDA.csv', index=False)
data_visualization.head()

,Rating,Company Name,Location,Headquarters,Size,Type of ownership,Industry,Sector,Revenue,job_simplified,seniority,Min Salary,Max Salary,Average Salary,Python_yn,Spark,AWS_yn,Excel_yn,Tableau_yn,SQL_yn,Rating Category,job_state,same_state,Age
0,3.8,Tecolote Research,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),data scientist,Other,53000,91000,72000.0,1,0,0,1,1,0,Medium Rating,NM,0,50
1,3.4,University of Maryland Medical System,"Linthicum, MD","Baltimore, MD",10000+ employees,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),data scientist,Other,63000,112000,87500.0,1,0,0,0,0,0,Medium Rating,MD,0,39
2,4.8,KnowBe4,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,Company - Private,Security Services,Business Services,$100 to $500 million (USD),data scientist,Other,80000,90000,85000.0,1,1,0,1,0,1,High Rating,FL,1,13
3,3.8,PNNL,"Richland, WA","Richland, WA",1001 to 5000 employees,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),data scientist,Other,56000,97000,76500.0,1,0,0,0,0,0,Medium Rating,WA,1,58
4,2.9,Affinity Solutions,"New York, NY","New York, NY",51 to 200 employees,Company - Private,Advertising & Marketing,Business Services,Unknown,data scientist,Other,86000,143000,114500.0,1,0,0,1,0,1,Low Rating,NY,1,25


In [51]:
data_visualization.info()

<class 'pandas.core.frame.DataFrame'>
Index: 742 entries, 0 to 955
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rating             742 non-null    float64
 1   Company Name       742 non-null    object 
 2   Location           742 non-null    object 
 3   Headquarters       742 non-null    object 
 4   Size               742 non-null    object 
 5   Type of ownership  742 non-null    object 
 6   Industry           742 non-null    object 
 7   Sector             742 non-null    object 
 8   Revenue            742 non-null    object 
 9   job_simplified     742 non-null    object 
 10  seniority          742 non-null    object 
 11  Min Salary         742 non-null    int64  
 12  Max Salary         742 non-null    int64  
 13  Average Salary     742 non-null    float64
 14  Python_yn          742 non-null    int64  
 15  Spark              742 non-null    int64  
 16  AWS_yn             742 non-null